In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

print("PyTorch version:", torch.__version__)

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)

PyTorch version: 2.14.0
Device: mps


In [2]:
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)

print("Random seed:", SEED)

Random seed: 42


In [3]:
train_data = np.load(
    "../data/processed/train_sequences.npz"
)

val_data = np.load(
    "../data/processed/validation_sequences.npz"
)

test_data = np.load(
    "../data/processed/test_sequences.npz"
)

X_train = train_data["X"]
y_train = train_data["y"]

X_val = val_data["X"]
y_val = val_data["y"]

X_test = test_data["X"]
y_test = test_data["y"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (23973, 12, 207)
y_train: (23973, 207)
X_val: (3410, 12, 207)
y_val: (3410, 207)
X_test: (6838, 12, 207)
y_test: (6838, 207)


In [4]:
train_target_mask = np.load(
    "../data/processed/train_target_mask.npy"
)

val_target_mask = np.load(
    "../data/processed/validation_target_mask.npy"
)

test_target_mask = np.load(
    "../data/processed/test_target_mask.npy"
)

print("Train mask:", train_target_mask.shape)
print("Validation mask:", val_target_mask.shape)

print("Test mask:", test_target_mask.shape)

Train mask: (23973, 207)
Validation mask: (3410, 207)
Test mask: (6838, 207)


In [7]:
adjacency_matrix = np.load(
    "../data/processed/adjacency_matrix.npy"
)

with open(
    "../data/processed/graph_sensor_ids.txt",
    "r"
) as f:
    graph_sensor_ids = [
        sensor_id.strip()
        for sensor_id in f.read().split(",")
        if sensor_id.strip()
    ]

print("Adjacency shape:", adjacency_matrix.shape)
print("Number of sensors:", len(graph_sensor_ids))
print("First 10 sensor IDs:", graph_sensor_ids[:10])
print("Contains NaN:", np.isnan(adjacency_matrix).any())
print("Contains Inf:", np.isinf(adjacency_matrix).any())

Adjacency shape: (207, 207)
Number of sensors: 207
First 10 sensor IDs: ['773869', '767541', '767542', '717447', '717446', '717445', '773062', '767620', '737529', '717816']
Contains NaN: False
Contains Inf: False


In [8]:
NUM_SENSORS = X_train.shape[2]
INPUT_STEPS = X_train.shape[1]
FORECAST_HORIZON = 6

assert NUM_SENSORS == 207
assert INPUT_STEPS == 12
assert adjacency_matrix.shape == (
    NUM_SENSORS,
    NUM_SENSORS
)

assert len(graph_sensor_ids) == NUM_SENSORS

print("Number of sensors:", NUM_SENSORS)
print("Historical time steps:", INPUT_STEPS)
print("Forecast horizon:", FORECAST_HORIZON)
print("Sensor ordering verified.")
print("All dimensions verified.")

Number of sensors: 207
Historical time steps: 12
Forecast horizon: 6
Sensor ordering verified.
All dimensions verified.


## UrbanFlow Architecture

UrbanFlow combines spatial and temporal deep learning.

The GNN branch learns spatial relationships between connected traffic sensors,
while the Transformer branch learns temporal traffic patterns across the
historical observation window.

The learned spatial and temporal representations are fused before predicting
traffic speed 30 minutes into the future.

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print("PyTorch version:", torch.__version__)
print("Device:", device)

PyTorch version: 2.14.0
Device: mps


In [12]:
class GraphConvolution(nn.Module):

    def __init__(self, in_features, out_features):
        super().__init__()

        self.linear = nn.Linear(
            in_features,
            out_features
        )

    def forward(self, x, adjacency):

        aggregated = torch.matmul(
            adjacency,
            x
        )

        output = self.linear(
            aggregated
        )

        return output

In [13]:
class SpatialGNNBlock(nn.Module):

    def __init__(self, num_sensors, hidden_size):
        super().__init__()

        self.gcn1 = GraphConvolution(
            in_features=1,
            out_features=hidden_size
        )

        self.gcn2 = GraphConvolution(
            in_features=hidden_size,
            out_features=hidden_size
        )

    def forward(self, x, adjacency):


        batch_size, time_steps, num_sensors = x.shape

        spatial_features = []

        for t in range(time_steps):

            snapshot = x[:, t, :]

            snapshot = snapshot.unsqueeze(-1)

            h = F.relu(
                self.gcn1(
                    snapshot,
                    adjacency
                )
            )

            h = F.relu(
                self.gcn2(
                    h,
                    adjacency
                )
            )

            spatial_features.append(h)

        spatial_features = torch.stack(
            spatial_features,
            dim=1
        )


        return spatial_features

In [14]:
class TemporalTransformerBlock(nn.Module):

    def __init__(
        self,
        hidden_size,
        num_heads=4,
        num_layers=2,
        dropout=0.2
    ):
        super().__init__()

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=num_heads,
            dim_feedforward=hidden_size * 2,
            dropout=dropout,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

    def forward(self, x):


        batch_size, time_steps, num_sensors, hidden_size = x.shape


        x = x.permute(0, 2, 1, 3)

        x = x.reshape(
            batch_size * num_sensors,
            time_steps,
            hidden_size
        )

        x = self.transformer(x)

        x = x[:, -1, :]


        x = x.reshape(
            batch_size,
            num_sensors,
            hidden_size
        )

        return x

In [15]:
class UrbanFlowModel(nn.Module):

    def __init__(
        self,
        num_sensors,
        hidden_size=64,
        num_heads=4,
        num_transformer_layers=2,
        dropout=0.2
    ):
        super().__init__()

        self.spatial_gnn = SpatialGNNBlock(
            num_sensors=num_sensors,
            hidden_size=hidden_size
        )

        self.temporal_transformer = TemporalTransformerBlock(
            hidden_size=hidden_size,
            num_heads=num_heads,
            num_layers=num_transformer_layers,
            dropout=dropout
        )

        self.fusion = nn.Sequential(
            nn.Linear(
                hidden_size * 2,
                hidden_size
            ),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.output_layer = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x, adjacency):


        spatial_features = self.spatial_gnn(
            x,
            adjacency
        )



        temporal_features = self.temporal_transformer(
            spatial_features
        )


        spatial_last = spatial_features[:, -1, :, :]


        fused_features = torch.cat(
            [
                spatial_last,
                temporal_features
            ],
            dim=-1
        )

        fused_features = self.fusion(
            fused_features
        )


        prediction = self.output_layer(
            fused_features
        )


        prediction = prediction.squeeze(-1)

        return prediction

In [16]:
URBANFLOW_HIDDEN = 64
URBANFLOW_HEADS = 4
URBANFLOW_LAYERS = 2
URBANFLOW_DROPOUT = 0.2

urbanflow_model = UrbanFlowModel(
    num_sensors=NUM_SENSORS,
    hidden_size=URBANFLOW_HIDDEN,
    num_heads=URBANFLOW_HEADS,
    num_transformer_layers=URBANFLOW_LAYERS,
    dropout=URBANFLOW_DROPOUT
).to(device)

print(urbanflow_model)

UrbanFlowModel(
  (spatial_gnn): SpatialGNNBlock(
    (gcn1): GraphConvolution(
      (linear): Linear(in_features=1, out_features=64, bias=True)
    )
    (gcn2): GraphConvolution(
      (linear): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (temporal_transformer): TemporalTransformerBlock(
    (transformer): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
          )
          (linear1): Linear(in_features=64, out_features=128, bias=True)
          (dropout): Dropout(p=0.2, inplace=False)
          (linear2): Linear(in_features=128, out_features=64, bias=True)
          (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
          (dropout1): Dropout(p=0.2, inplace=False)
        

In [17]:
total_parameters = sum(
    parameter.numel()
    for parameter in urbanflow_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in urbanflow_model.parameters()
    if parameter.requires_grad
)

print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

Total parameters: 79553
Trainable parameters: 79553


In [20]:

A = torch.tensor(
    adjacency_matrix,
    dtype=torch.float32,
    device=device
)

degree = A.sum(dim=1)

degree_inv_sqrt = torch.pow(
    degree.clamp(min=1e-8),
    -0.5
)

D_inv_sqrt = torch.diag(
    degree_inv_sqrt
)


A_normalized = (
    D_inv_sqrt
    @ A
    @ D_inv_sqrt
)

print("Normalized adjacency shape:", A_normalized.shape)
print("Device:", A_normalized.device)
print("Contains NaN:", torch.isnan(A_normalized).any().item())
print("Contains Inf:", torch.isinf(A_normalized).any().item())

Normalized adjacency shape: torch.Size([207, 207])
Device: mps:0
Contains NaN: False
Contains Inf: False


In [21]:
sample_input = torch.tensor(
    X_train[:8],
    dtype=torch.float32,
    device=device
)

sample_output = urbanflow_model(
    sample_input,
    A_normalized
)

print("Input shape:", sample_input.shape)
print("Output shape:", sample_output.shape)
print("Expected output shape:", (8, NUM_SENSORS))

Input shape: torch.Size([8, 12, 207])
Output shape: torch.Size([8, 207])
Expected output shape: (8, 207)


In [22]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 64

train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
    torch.tensor(train_target_mask, dtype=torch.bool)
)

val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32),
    torch.tensor(val_target_mask, dtype=torch.bool)
)

test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32),
    torch.tensor(test_target_mask, dtype=torch.bool)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Training batches: 375
Validation batches: 54
Test batches: 107


In [23]:
def masked_mse_loss(predictions, targets, mask):

    squared_error = (predictions - targets) ** 2

    masked_error = squared_error * mask.float()

    return (
        masked_error.sum()
        / mask.sum().clamp_min(1)
    )

In [24]:
criterion = masked_mse_loss

optimizer = torch.optim.Adam(
    urbanflow_model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

print(optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 1e-05
)


In [25]:
EPOCHS = 20
PATIENCE = 5

best_val_loss = float("inf")
patience_counter = 0

train_history = []
val_history = []

best_model_path = "../models/urbanflow_best.pth"

In [26]:
for epoch in range(EPOCHS):

    urbanflow_model.train()

    total_train_loss = 0.0
    train_batches = 0

    for batch_x, batch_y, batch_mask in train_loader:

        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        batch_mask = batch_mask.to(device)

        optimizer.zero_grad()

        predictions = urbanflow_model(
            batch_x,
            A_normalized
        )

        loss = criterion(
            predictions,
            batch_y,
            batch_mask
        )

        loss.backward()

        optimizer.step()

        total_train_loss += loss.item()
        train_batches += 1

    avg_train_loss = (
        total_train_loss / train_batches
    )


    urbanflow_model.eval()

    total_val_loss = 0.0
    val_batches = 0

    with torch.no_grad():

        for batch_x, batch_y, batch_mask in val_loader:

            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            batch_mask = batch_mask.to(device)

            predictions = urbanflow_model(
                batch_x,
                A_normalized
            )

            loss = criterion(
                predictions,
                batch_y,
                batch_mask
            )

            total_val_loss += loss.item()
            val_batches += 1

    avg_val_loss = (
        total_val_loss / val_batches
    )

    train_history.append(avg_train_loss)
    val_history.append(avg_val_loss)

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train Loss: {avg_train_loss:.6f} | "
        f"Val Loss: {avg_val_loss:.6f}"
    )


    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        torch.save(
            urbanflow_model.state_dict(),
            best_model_path
        )

        patience_counter = 0

        print("  → Best UrbanFlow model saved.")

    else:

        patience_counter += 1

        if patience_counter >= PATIENCE:

            print("  → Early stopping triggered.")
            break

Epoch 01/20 | Train Loss: 0.686382 | Val Loss: 0.645453
  → Best UrbanFlow model saved.
Epoch 02/20 | Train Loss: 0.658543 | Val Loss: 0.634650
  → Best UrbanFlow model saved.
Epoch 03/20 | Train Loss: 0.649079 | Val Loss: 0.622347
  → Best UrbanFlow model saved.
Epoch 04/20 | Train Loss: 0.643793 | Val Loss: 0.620932
  → Best UrbanFlow model saved.
Epoch 05/20 | Train Loss: 0.639646 | Val Loss: 0.616070
  → Best UrbanFlow model saved.
Epoch 06/20 | Train Loss: 0.636895 | Val Loss: 0.618842
Epoch 07/20 | Train Loss: 0.633668 | Val Loss: 0.609484
  → Best UrbanFlow model saved.
Epoch 08/20 | Train Loss: 0.631343 | Val Loss: 0.610325
Epoch 09/20 | Train Loss: 0.628619 | Val Loss: 0.609670
Epoch 10/20 | Train Loss: 0.626568 | Val Loss: 0.606388
  → Best UrbanFlow model saved.
Epoch 11/20 | Train Loss: 0.625208 | Val Loss: 0.609482
Epoch 12/20 | Train Loss: 0.623718 | Val Loss: 0.599700
  → Best UrbanFlow model saved.
Epoch 13/20 | Train Loss: 0.621587 | Val Loss: 0.601888
Epoch 14/20 | Tr

In [27]:
urbanflow_model.load_state_dict(
    torch.load(
        "../models/urbanflow_best.pth",
        map_location=device
    )
)

urbanflow_model.to(device)
urbanflow_model.eval()

print("Best UrbanFlow model restored.")
print("Best validation loss:", best_val_loss)

Best UrbanFlow model restored.
Best validation loss: 0.592893522646692


In [28]:
urbanflow_val_predictions = []
urbanflow_val_targets = []
urbanflow_val_masks = []

with torch.no_grad():

    for batch_x, batch_y, batch_mask in val_loader:

        batch_x = batch_x.to(device)

        predictions = urbanflow_model(
            batch_x,
            A_normalized
        )

        urbanflow_val_predictions.append(
            predictions.cpu().numpy()
        )

        urbanflow_val_targets.append(
            batch_y.numpy()
        )

        urbanflow_val_masks.append(
            batch_mask.numpy()
        )

urbanflow_val_predictions = np.concatenate(
    urbanflow_val_predictions,
    axis=0
)

urbanflow_val_targets = np.concatenate(
    urbanflow_val_targets,
    axis=0
)

urbanflow_val_masks = np.concatenate(
    urbanflow_val_masks,
    axis=0
)

print(
    "UrbanFlow validation predictions:",
    urbanflow_val_predictions.shape
)

print(
    "UrbanFlow validation targets:",
    urbanflow_val_targets.shape
)

UrbanFlow validation predictions: (3410, 207)
UrbanFlow validation targets: (3410, 207)


In [31]:
import joblib

scaler = joblib.load(
    "../data/processed/traffic_scaler.pkl"
)

urbanflow_val_predictions_mph = scaler.inverse_transform(
    urbanflow_val_predictions
)

urbanflow_val_targets_mph = scaler.inverse_transform(
    urbanflow_val_targets
)

print(
    "Prediction range:",
    urbanflow_val_predictions_mph.min(),
    "to",
    urbanflow_val_predictions_mph.max(),
    "mph"
)

print(
    "Target range:",
    urbanflow_val_targets_mph.min(),
    "to",
    urbanflow_val_targets_mph.max(),
    "mph"
)

Prediction range: -41.686073 to 75.412186 mph
Target range: 0.4999962 to 70.0 mph


In [32]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mask = urbanflow_val_masks.astype(bool)

pred_valid = urbanflow_val_predictions_mph[mask]
target_valid = urbanflow_val_targets_mph[mask]

urbanflow_val_mae = mean_absolute_error(
    target_valid,
    pred_valid
)

urbanflow_val_rmse = np.sqrt(
    mean_squared_error(
        target_valid,
        pred_valid
    )
)

print("UrbanFlow Validation MAE:", urbanflow_val_mae)
print("UrbanFlow Validation RMSE:", urbanflow_val_rmse)

UrbanFlow Validation MAE: 4.305187225341797
UrbanFlow Validation RMSE: 7.42307813027875


In [33]:
comparison_results = pd.DataFrame({
    "model": [
        "Persistence",
        "Linear Regression",
        "Random Forest",
        "LSTM",
        "GNN",
        "Transformer",
        "UrbanFlow"
    ],
    "validation_mae": [
        3.8050,
        3.9204,
        3.7433,
        3.5952,
        4.715280055999756,
        3.4215071201324463,
        urbanflow_val_mae
    ],
    "validation_rmse": [
        7.4818,
        6.4007,
        6.7961,
        6.4763,
        7.856982737309215,
        6.177782230390839,
        urbanflow_val_rmse
    ]
})

comparison_results

,model,validation_mae,validation_rmse
0,Persistence,3.805000,7.481800
1,Linear Regression,3.920400,6.400700
2,Random Forest,3.743300,6.796100
3,LSTM,3.595200,6.476300
4,GNN,4.715280,7.856983
5,Transformer,3.421507,6.177782
6,UrbanFlow,4.305187,7.423078


In [34]:
comparison_results.to_csv(
    "../results/model_validation_comparison.csv",
    index=False
)

print(
    "Saved to ../results/model_validation_comparison.csv"
)

Saved to ../results/model_validation_comparison.csv


In [35]:
print("UrbanFlow prediction statistics")
print("--------------------------------")

print(
    "Mean prediction:",
    urbanflow_val_predictions_mph.mean()
)

print(
    "Mean target:",
    urbanflow_val_targets_mph.mean()
)

print(
    "Prediction std:",
    urbanflow_val_predictions_mph.std()
)

print(
    "Target std:",
    urbanflow_val_targets_mph.std()
)

negative_count = np.sum(
    urbanflow_val_predictions_mph < 0
)

above_70_count = np.sum(
    urbanflow_val_predictions_mph > 70
)

total_predictions = urbanflow_val_predictions_mph.size

print(
    "Negative predictions:",
    negative_count
)

print(
    "Negative percentage:",
    100 * negative_count / total_predictions
)

print(
    "Above 70 mph:",
    above_70_count
)

print(
    "Above 70 percentage:",
    100 * above_70_count / total_predictions
)

UrbanFlow prediction statistics
--------------------------------
Mean prediction: 59.145145
Mean target: 59.182274
Prediction std: 10.759787
Target std: 12.769781
Negative predictions: 710
Negative percentage: 0.10058509357247085
Above 70 mph: 11551
Above 70 percentage: 1.6364203040219871


In [36]:
absolute_errors = np.abs(
    urbanflow_val_predictions_mph -
    urbanflow_val_targets_mph
)

mask = urbanflow_val_masks.astype(bool)

errors = absolute_errors[mask]
targets = urbanflow_val_targets_mph[mask]

speed_bins = [0, 20, 40, 60, 70, np.inf]
speed_labels = [
    "0-20 mph",
    "20-40 mph",
    "40-60 mph",
    "60-70 mph",
    "70+ mph"
]

speed_categories = pd.cut(
    targets,
    bins=speed_bins,
    labels=speed_labels,
    include_lowest=True
)

error_analysis = pd.DataFrame({
    "target_speed": targets,
    "absolute_error": errors,
    "speed_range": speed_categories
})

error_summary = (
    error_analysis
    .groupby("speed_range", observed=False)
    .agg(
        samples=("absolute_error", "count"),
        mean_absolute_error=("absolute_error", "mean")
    )
    .reset_index()
)

print(error_summary)

  speed_range  samples  mean_absolute_error
0    0-20 mph    20467            17.211613
1   20-40 mph    48895            11.233978
2   40-60 mph   138884             4.965521
3   60-70 mph   456598             2.783830
4     70+ mph        0                  NaN


In [37]:
transformer_results = pd.read_csv(
    "../results/transformer_validation_results.csv"
)

gnn_results = pd.read_csv(
    "../results/gnn_validation_results.csv"
)

print("Transformer:")
print(transformer_results)

print("\nGNN:")
print(gnn_results)

print("\nUrbanFlow:")
print(
    comparison_results[
        comparison_results["model"] == "UrbanFlow"
    ]
)

Transformer:
         model  validation_mae  validation_rmse
0  Transformer        3.421507         6.177782

GNN:
  model  validation_mae  validation_rmse
0   GNN         4.71528         7.856983

UrbanFlow:
       model  validation_mae  validation_rmse
6  UrbanFlow        4.305187         7.423078


In [38]:
error_summary.to_csv(
    "../results/urbanflow_error_by_speed.csv",
    index=False
)

print(
    "Saved to ../results/urbanflow_error_by_speed.csv"
)

Saved to ../results/urbanflow_error_by_speed.csv


In [39]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class TemporalTransformerBranch(nn.Module):

    def __init__(
        self,
        num_sensors,
        hidden_size=128,
        num_heads=4,
        num_layers=2,
        dropout=0.2
    ):
        super().__init__()

        self.num_sensors = num_sensors
        self.hidden_size = hidden_size

        self.input_projection = nn.Linear(
            1,
            hidden_size
        )

        self.positional_encoding = nn.Parameter(
            torch.zeros(1, INPUT_STEPS, hidden_size)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=num_heads,
            dim_feedforward=hidden_size * 2,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

    def forward(self, x):

        batch_size = x.size(0)

        x = x.permute(0, 2, 1)

        x = x.reshape(
            batch_size * self.num_sensors,
            INPUT_STEPS,
            1
        )

        x = self.input_projection(x)

        x = x + self.positional_encoding

        x = self.transformer(x)

        x = x[:, -1, :]

        x = x.reshape(
            batch_size,
            self.num_sensors,
            self.hidden_size
        )

        return x

In [40]:
class SpatialGNNBranch(nn.Module):

    def __init__(
        self,
        hidden_size=128
    ):
        super().__init__()

        self.gcn1 = nn.Linear(
            1,
            hidden_size
        )

        self.gcn2 = nn.Linear(
            hidden_size,
            hidden_size
        )

    def graph_convolution(
        self,
        x,
        adjacency,
        layer
    ):


        aggregated = torch.matmul(
            adjacency,
            x
        )

        return layer(aggregated)

    def forward(
        self,
        x,
        adjacency
    ):

        snapshot = x[:, -1, :]

        snapshot = snapshot.unsqueeze(-1)

        h = self.graph_convolution(
            snapshot,
            adjacency,
            self.gcn1
        )

        h = F.relu(h)

        h = self.graph_convolution(
            h,
            adjacency,
            self.gcn2
        )

        h = F.relu(h)

        return h

In [41]:
class UrbanFlowV2(nn.Module):

    def __init__(
        self,
        num_sensors,
        hidden_size=128,
        num_heads=4,
        num_layers=2,
        dropout=0.2
    ):
        super().__init__()

        self.temporal_branch = TemporalTransformerBranch(
            num_sensors=num_sensors,
            hidden_size=hidden_size,
            num_heads=num_heads,
            num_layers=num_layers,
            dropout=dropout
        )

        self.spatial_branch = SpatialGNNBranch(
            hidden_size=hidden_size
        )

        self.spatial_gate = nn.Parameter(
            torch.tensor(0.1)
        )

        self.fusion = nn.Sequential(
            nn.Linear(
                hidden_size * 2,
                hidden_size
            ),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.output_layer = nn.Linear(
            hidden_size,
            1
        )

    def forward(
        self,
        x,
        adjacency
    ):


        temporal_features = self.temporal_branch(x)


        spatial_features = self.spatial_branch(
            x,
            adjacency
        )

        spatial_features = (
            self.spatial_gate *
            spatial_features
        )


        combined = torch.cat(
            [
                temporal_features,
                spatial_features
            ],
            dim=-1
        )

        fused = self.fusion(
            combined
        )


        output = self.output_layer(
            fused
        ).squeeze(-1)

        return output

In [42]:
urbanflow_v2 = UrbanFlowV2(
    num_sensors=NUM_SENSORS,
    hidden_size=128,
    num_heads=4,
    num_layers=2,
    dropout=0.2
).to(device)

print(urbanflow_v2)

UrbanFlowV2(
  (temporal_branch): TemporalTransformerBranch(
    (input_projection): Linear(in_features=1, out_features=128, bias=True)
    (transformer): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (linear1): Linear(in_features=128, out_features=256, bias=True)
          (dropout): Dropout(p=0.2, inplace=False)
          (linear2): Linear(in_features=256, out_features=128, bias=True)
          (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
          (dropout1): Dropout(p=0.2, inplace=False)
          (dropout2): Dropout(p=0.2, inplace=False)
        )
      )
    )
  )
  (spatial_branch): SpatialGNNBranch(
    (gcn1): Linear(in_features=1, out_features=128, bias=Tr

In [43]:
total_parameters_v2 = sum(
    parameter.numel()
    for parameter in urbanflow_v2.parameters()
)

trainable_parameters_v2 = sum(
    parameter.numel()
    for parameter in urbanflow_v2.parameters()
    if parameter.requires_grad
)

print(
    "Total parameters:",
    total_parameters_v2
)

print(
    "Trainable parameters:",
    trainable_parameters_v2
)

Total parameters: 316546
Trainable parameters: 316546


In [44]:
sample_input = torch.tensor(
    X_train[:8],
    dtype=torch.float32,
    device=device
)

sample_output = urbanflow_v2(
    sample_input,
    A_normalized
)

print(
    "Input shape:",
    sample_input.shape
)

print(
    "Output shape:",
    sample_output.shape
)

print(
    "Expected output shape:",
    (8, NUM_SENSORS)
)

Input shape: torch.Size([8, 12, 207])
Output shape: torch.Size([8, 207])
Expected output shape: (8, 207)


In [45]:
criterion_v2 = nn.MSELoss()

optimizer_v2 = torch.optim.Adam(
    urbanflow_v2.parameters(),
    lr=0.001
)

EPOCHS = 20
PATIENCE = 5

best_val_loss_v2 = float("inf")
patience_counter_v2 = 0

train_history_v2 = []
val_history_v2 = []

best_model_path_v2 = "../models/urbanflow_v2_best.pth"

for epoch in range(EPOCHS):


    urbanflow_v2.train()

    train_losses = []

    for batch_x, batch_y, batch_mask in train_loader:

        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        batch_mask = batch_mask.to(device)

        optimizer_v2.zero_grad()

        predictions = urbanflow_v2(
            batch_x,
            A_normalized
        )

        loss = masked_mse_loss(
            predictions,
            batch_y,
            batch_mask
        )

        loss.backward()

        optimizer_v2.step()

        train_losses.append(
            loss.item()
        )

    avg_train_loss = np.mean(
        train_losses
    )


    urbanflow_v2.eval()

    val_losses = []

    with torch.no_grad():

        for batch_x, batch_y, batch_mask in val_loader:

            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            batch_mask = batch_mask.to(device)

            predictions = urbanflow_v2(
                batch_x,
                A_normalized
            )

            loss = masked_mse_loss(
                predictions,
                batch_y,
                batch_mask
            )

            val_losses.append(
                loss.item()
            )

    avg_val_loss = np.mean(
        val_losses
    )

    train_history_v2.append(
        avg_train_loss
    )

    val_history_v2.append(
        avg_val_loss
    )

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train Loss: {avg_train_loss:.6f} | "
        f"Val Loss: {avg_val_loss:.6f}"
    )


    if avg_val_loss < best_val_loss_v2:

        best_val_loss_v2 = avg_val_loss
        patience_counter_v2 = 0

        torch.save(
            urbanflow_v2.state_dict(),
            best_model_path_v2
        )

        print(" → Best UrbanFlow v2 model saved.")

    else:

        patience_counter_v2 += 1

        if patience_counter_v2 >=  PATIENCE:

            print(" → Early stopping triggered.")
            break

Epoch 01/20 | Train Loss: 0.548178 | Val Loss: 0.527494
 → Best UrbanFlow v2 model saved.
Epoch 02/20 | Train Loss: 0.532108 | Val Loss: 0.515512
 → Best UrbanFlow v2 model saved.
Epoch 03/20 | Train Loss: 0.529885 | Val Loss: 0.513142
 → Best UrbanFlow v2 model saved.
Epoch 04/20 | Train Loss: 0.527496 | Val Loss: 0.532730
Epoch 05/20 | Train Loss: 0.526109 | Val Loss: 0.512634
 → Best UrbanFlow v2 model saved.
Epoch 06/20 | Train Loss: 0.525708 | Val Loss: 0.513010
Epoch 07/20 | Train Loss: 0.524264 | Val Loss: 0.514466
Epoch 08/20 | Train Loss: 0.524753 | Val Loss: 0.511071
 → Best UrbanFlow v2 model saved.
Epoch 09/20 | Train Loss: 0.524030 | Val Loss: 0.507621
 → Best UrbanFlow v2 model saved.
Epoch 10/20 | Train Loss: 0.522708 | Val Loss: 0.507436
 → Best UrbanFlow v2 model saved.
Epoch 11/20 | Train Loss: 0.521583 | Val Loss: 0.519329
Epoch 12/20 | Train Loss: 0.521159 | Val Loss: 0.513574
Epoch 13/20 | Train Loss: 0.520747 | Val Loss: 0.507457
Epoch 14/20 | Train Loss: 0.519825

In [46]:
urbanflow_v2.load_state_dict(
    torch.load(
        best_model_path_v2,
        map_location=device
    )
)

urbanflow_v2.to(device)
urbanflow_v2.eval()

print("Best UrbanFlow v2 model restored.")
print(
    "Best validation loss:",
    best_val_loss_v2
)

Best UrbanFlow v2 model restored.
Best validation loss: 0.503111020144489


In [47]:
urbanflow_v2_history = pd.DataFrame({
    "epoch": range(
        1,
        len(train_history_v2) + 1
    ),
    "train_loss": train_history_v2,
    "validation_loss": val_history_v2
})

urbanflow_v2_history.to_csv(
    "../results/urbanflow_v2_training_history.csv",
    index=False
)

print(
    "Saved to ../results/urbanflow_v2_training_history.csv"
)

Saved to ../results/urbanflow_v2_training_history.csv


In [48]:
urbanflow_v2.eval()

urbanflow_v2_val_predictions = []

with torch.no_grad():

    for batch_x, batch_y, batch_mask in val_loader:

        batch_x = batch_x.to(device)

        predictions = urbanflow_v2(
            batch_x,
            A_normalized
        )

        urbanflow_v2_val_predictions.append(
            predictions.cpu().numpy()
        )

urbanflow_v2_val_predictions = np.concatenate(
    urbanflow_v2_val_predictions,
    axis=0
)

print(
    "UrbanFlow v2 validation predictions:",
    urbanflow_v2_val_predictions.shape
)

print(
    "Expected shape:",
    y_val.shape
)

UrbanFlow v2 validation predictions: (3410, 207)
Expected shape: (3410, 207)


In [49]:
import joblib

scaler = joblib.load(
    "../data/processed/traffic_scaler.pkl"
)

urbanflow_v2_val_predictions_mph = scaler.inverse_transform(
    urbanflow_v2_val_predictions
)

urbanflow_v2_val_targets_mph = scaler.inverse_transform(
    y_val
)

print(
    "Prediction range:",
    urbanflow_v2_val_predictions_mph.min(),
    "to",
    urbanflow_v2_val_predictions_mph.max(),
    "mph"
)

print(
    "Target range:",
    urbanflow_v2_val_targets_mph.min(),
    "to",
    urbanflow_v2_val_targets_mph.max(),
    "mph"
)

Prediction range: -0.8463669 to 70.85659 mph
Target range: 0.4999962 to 70.0 mph


In [50]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mask = val_target_mask.astype(bool)

y_true = urbanflow_v2_val_targets_mph[mask]
y_pred = urbanflow_v2_val_predictions_mph[mask]

urbanflow_v2_val_mae = mean_absolute_error(
    y_true,
    y_pred
)

urbanflow_v2_val_rmse = np.sqrt(
    mean_squared_error(
        y_true,
        y_pred
    )
)

print(
    "UrbanFlow v2 Validation MAE:",
    urbanflow_v2_val_mae
)

print(
    "UrbanFlow v2 Validation RMSE:",
    urbanflow_v2_val_rmse
)

UrbanFlow v2 Validation MAE: 3.514186143875122
UrbanFlow v2 Validation RMSE: 6.68833732710693


In [51]:
urbanflow_v2_results = pd.DataFrame({
    "model": ["UrbanFlow_v2"],
    "validation_mae": [urbanflow_v2_val_mae],
    "validation_rmse": [urbanflow_v2_val_rmse]
})

urbanflow_v2_results.to_csv(
    "../results/urbanflow_v2_validation_results.csv",
    index=False
)

print(urbanflow_v2_results)
print("\nSaved to ../results/urbanflow_v2_validation_results.csv")

          model  validation_mae  validation_rmse
0  UrbanFlow_v2        3.514186         6.688337

Saved to ../results/urbanflow_v2_validation_results.csv


In [52]:
baseline_results = pd.read_csv(
    "../results/baseline_validation_results.csv"
)

lstm_results = pd.read_csv(
    "../results/lstm_validation_results.csv"
)

gnn_results = pd.read_csv(
    "../results/gnn_validation_results.csv"
)

transformer_results = pd.read_csv(
    "../results/transformer_validation_results.csv"
)

urbanflow_v1_results = pd.DataFrame({
    "model": ["UrbanFlow_v1"],
    "validation_mae": [4.305187225341797],
    "validation_rmse": [7.42307813027875]
})

all_model_results = pd.concat(
    [
        baseline_results,
        lstm_results,
        gnn_results,
        transformer_results,
        urbanflow_v1_results,
        urbanflow_v2_results
    ],
    ignore_index=True
)

print(all_model_results)

               Model  Validation_MAE  Validation_RMSE         model  \
0        Persistence        3.804962         7.481793           NaN   
1  Linear Regression        3.920410         6.400667           NaN   
2      Random Forest        3.743304         6.796092           NaN   
3               LSTM        3.595160         6.476273           NaN   
4                NaN             NaN              NaN           GNN   
5                NaN             NaN              NaN   Transformer   
6                NaN             NaN              NaN  UrbanFlow_v1   
7                NaN             NaN              NaN  UrbanFlow_v2   

   validation_mae  validation_rmse  
0             NaN              NaN  
1             NaN              NaN  
2             NaN              NaN  
3             NaN              NaN  
4        4.715280         7.856983  
5        3.421507         6.177782  
6        4.305187         7.423078  
7        3.514186         6.688337  


In [53]:
all_model_results_sorted = all_model_results.sort_values(
    by="validation_mae"
).reset_index(drop=True)

print(all_model_results_sorted)

               Model  Validation_MAE  Validation_RMSE         model  \
0                NaN             NaN              NaN   Transformer   
1                NaN             NaN              NaN  UrbanFlow_v2   
2                NaN             NaN              NaN  UrbanFlow_v1   
3                NaN             NaN              NaN           GNN   
4        Persistence        3.804962         7.481793           NaN   
5  Linear Regression        3.920410         6.400667           NaN   
6      Random Forest        3.743304         6.796092           NaN   
7               LSTM        3.595160         6.476273           NaN   

   validation_mae  validation_rmse  
0        3.421507         6.177782  
1        3.514186         6.688337  
2        4.305187         7.423078  
3        4.715280         7.856983  
4             NaN              NaN  
5             NaN              NaN  
6             NaN              NaN  
7             NaN              NaN  


In [54]:
all_model_results_sorted.to_csv(
    "../results/final_model_comparison.csv",
    index=False
)

print(
    "Saved to ../results/final_model_comparison.csv"
)

Saved to ../results/final_model_comparison.csv


In [55]:
error_analysis = pd.DataFrame({
    "actual_speed": y_true,
    "prediction": y_pred
})

error_analysis["absolute_error"] = (
    np.abs(
        error_analysis["actual_speed"]
        - error_analysis["prediction"]
    )
)

error_analysis["speed_range"] = pd.cut(
    error_analysis["actual_speed"],
    bins=[0, 20, 40, 60, 70],
    labels=[
        "0-20 mph",
        "20-40 mph",
        "40-60 mph",
        "60-70 mph"
    ],
    include_lowest=True
)

speed_error_summary = (
    error_analysis
    .groupby("speed_range", observed=False)
    .agg(
        samples=("absolute_error", "size"),
        mean_absolute_error=("absolute_error", "mean")
    )
    .reset_index()
)

print(speed_error_summary)

  speed_range  samples  mean_absolute_error
0    0-20 mph    20467            12.692064
1   20-40 mph    48895             9.347245
2   40-60 mph   138884             4.550428
3   60-70 mph   456598             2.162956


In [56]:
speed_error_summary.to_csv(
    "../results/urbanflow_v2_error_by_speed.csv",
    index=False
)

print(
    "Saved to ../results/urbanflow_v2_error_by_speed.csv"
)

Saved to ../results/urbanflow_v2_error_by_speed.csv


In [57]:

baseline_results = pd.read_csv(
    "../results/baseline_validation_results.csv"
)

lstm_results = pd.read_csv(
    "../results/lstm_validation_results.csv"
)

gnn_results = pd.read_csv(
    "../results/gnn_validation_results.csv"
)

transformer_results = pd.read_csv(
    "../results/transformer_validation_results.csv"
)

def standardize_result_columns(df):
    df = df.copy()

    df = df.rename(columns={
        "Model": "model",
        "Validation_MAE": "validation_mae",
        "Validation_RMSE": "validation_rmse"
    })

    return df[
        ["model", "validation_mae", "validation_rmse"]
    ]


baseline_results = standardize_result_columns(baseline_results)
lstm_results = standardize_result_columns(lstm_results)
gnn_results = standardize_result_columns(gnn_results)
transformer_results = standardize_result_columns(transformer_results)

urbanflow_v1_results = pd.DataFrame({
    "model": ["UrbanFlow_v1"],
    "validation_mae": [4.305187225341797],
    "validation_rmse": [7.42307813027875]
})

urbanflow_v2_results = standardize_result_columns(
    urbanflow_v2_results
)

all_model_results = pd.concat(
    [
        baseline_results,
        lstm_results,
        gnn_results,
        transformer_results,
        urbanflow_v1_results,
        urbanflow_v2_results
    ],
    ignore_index=True
)

print(all_model_results)

               model  validation_mae  validation_rmse
0        Persistence        3.804962         7.481793
1  Linear Regression        3.920410         6.400667
2      Random Forest        3.743304         6.796092
3               LSTM        3.595160         6.476273
4                GNN        4.715280         7.856983
5        Transformer        3.421507         6.177782
6       UrbanFlow_v1        4.305187         7.423078
7       UrbanFlow_v2        3.514186         6.688337


In [58]:
all_model_results_sorted = all_model_results.sort_values(
    by="validation_mae",
    ascending=True
).reset_index(drop=True)

print(all_model_results_sorted)

               model  validation_mae  validation_rmse
0        Transformer        3.421507         6.177782
1       UrbanFlow_v2        3.514186         6.688337
2               LSTM        3.595160         6.476273
3      Random Forest        3.743304         6.796092
4        Persistence        3.804962         7.481793
5  Linear Regression        3.920410         6.400667
6       UrbanFlow_v1        4.305187         7.423078
7                GNN        4.715280         7.856983


In [59]:
all_model_results_sorted.to_csv(
    "../results/final_model_comparison.csv",
    index=False
)

print("Saved clean final model comparison.")

Saved clean final model comparison.


In [60]:

test_data = np.load(
    "../data/processed/test_sequences.npz"
)

X_test = test_data["X"]
y_test = test_data["y"]

test_target_mask = np.load(
    "../data/processed/test_target_mask.npy"
)

print("Test input shape:", X_test.shape)
print("Test target shape:", y_test.shape)
print("Test mask shape:", test_target_mask.shape)

Test input shape: (6838, 12, 207)
Test target shape: (6838, 207)
Test mask shape: (6838, 207)


In [61]:
from torch.utils.data import TensorDataset, DataLoader

test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32),
    torch.tensor(test_target_mask, dtype=torch.bool)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

print("Test batches:", len(test_loader))

Test batches: 107


In [62]:
best_model_path_v2 = "../models/urbanflow_v2_best.pth"

urbanflow_v2.load_state_dict(
    torch.load(
        best_model_path_v2,
        map_location=device
    )
)

urbanflow_v2.to(device)
urbanflow_v2.eval()

print("Best UrbanFlow v2 model loaded.")

Best UrbanFlow v2 model loaded.


In [63]:
urbanflow_v2_test_predictions = []

with torch.no_grad():

    for batch_x, batch_y, batch_mask in test_loader:

        batch_x = batch_x.to(device)

        predictions = urbanflow_v2(
            batch_x,
            A_normalized
        )

        urbanflow_v2_test_predictions.append(
            predictions.cpu().numpy()
        )

urbanflow_v2_test_predictions = np.concatenate(
    urbanflow_v2_test_predictions,
    axis=0
)

print(
    "UrbanFlow v2 test predictions:",
    urbanflow_v2_test_predictions.shape
)

print(
    "Expected:",
    y_test.shape
)

UrbanFlow v2 test predictions: (6838, 207)
Expected: (6838, 207)


In [64]:
urbanflow_v2_test_predictions_mph = scaler.inverse_transform(
    urbanflow_v2_test_predictions
)

urbanflow_v2_test_targets_mph = scaler.inverse_transform(
    y_test
)

print(
    "Prediction range:",
    urbanflow_v2_test_predictions_mph.min(),
    "to",
    urbanflow_v2_test_predictions_mph.max(),
    "mph"
)

print(
    "Target range:",
    urbanflow_v2_test_targets_mph.min(),
    "to",
    urbanflow_v2_test_targets_mph.max(),
    "mph"
)

Prediction range: -2.2423859 to 71.69493 mph
Target range: 0.75 to 70.0 mph


In [65]:
test_mask = test_target_mask.astype(bool)

test_y_true = urbanflow_v2_test_targets_mph[test_mask]
test_y_pred = urbanflow_v2_test_predictions_mph[test_mask]

urbanflow_v2_test_mae = mean_absolute_error(
    test_y_true,
    test_y_pred
)

urbanflow_v2_test_rmse = np.sqrt(
    mean_squared_error(
        test_y_true,
        test_y_pred
    )
)

print(
    "======================================"
)

print(
    "UrbanFlow v2 FINAL TEST MAE:",
    urbanflow_v2_test_mae,
    "mph"
)

print(
    "UrbanFlow v2 FINAL TEST RMSE:",
    urbanflow_v2_test_rmse,
    "mph"
)

print(
    "Evaluated test observations:",
    test_mask.sum()
)

UrbanFlow v2 FINAL TEST MAE: 3.908057689666748 mph
UrbanFlow v2 FINAL TEST RMSE: 7.208877484137099 mph
Evaluated test observations: 1266610


In [66]:
final_test_results = pd.DataFrame({
    "model": ["UrbanFlow_v2"],
    "test_mae": [urbanflow_v2_test_mae],
    "test_rmse": [urbanflow_v2_test_rmse],
    "evaluated_observations": [int(test_mask.sum())]
})

final_test_results.to_csv(
    "../results/urbanflow_v2_final_test_results.csv",
    index=False
)

print(final_test_results)

print(
    "\nSaved to ../results/urbanflow_v2_final_test_results.csv"
)

          model  test_mae  test_rmse  evaluated_observations
0  UrbanFlow_v2  3.908058   7.208877                 1266610

Saved to ../results/urbanflow_v2_final_test_results.csv


In [3]:
import numpy as np
import pandas as pd

In [6]:
print("urbanflow_v2:", "urbanflow_v2" in globals())
print("test_y_pred:", "test_y_pred" in globals())
print("test_y_true:", "test_y_true" in globals())
print("X_test:", "X_test" in globals())
print("y_test:", "y_test" in globals())
print("test_target_mask:", "test_target_mask" in globals())
print("scaler:", "scaler" in globals())
print("device:", "device" in globals())

urbanflow_v2: False
test_y_pred: False
test_y_true: False
X_test: False
y_test: False
test_target_mask: False
scaler: False
device: False


In [7]:
import pandas as pd

final_comparison = pd.DataFrame({
    "Model": [
        "Persistence",
        "Linear Regression",
        "Random Forest",
        "LSTM",
        "GNN",
        "Transformer",
        "UrbanFlow v1",
        "UrbanFlow v2"
    ],
    
    "Validation_MAE": [
        3.804962,
        3.920410,
        3.743304,
        3.595160,
        4.715280,
        3.421507,
        4.305187,
        3.514186
    ],
    
    "Validation_RMSE": [
        7.481793,
        6.400667,
        6.796092,
        6.476273,
        7.856983,
        6.177782,
        7.423078,
        6.688337
    ]
})

print(final_comparison)

final_comparison.to_csv(
    "../results/final_model_comparison_clean.csv",
    index=False
)

print("\nSaved to ../results/final_model_comparison_clean.csv")

               Model  Validation_MAE  Validation_RMSE
0        Persistence        3.804962         7.481793
1  Linear Regression        3.920410         6.400667
2      Random Forest        3.743304         6.796092
3               LSTM        3.595160         6.476273
4                GNN        4.715280         7.856983
5        Transformer        3.421507         6.177782
6       UrbanFlow v1        4.305187         7.423078
7       UrbanFlow v2        3.514186         6.688337

Saved to ../results/final_model_comparison_clean.csv


In [8]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [9]:
X_val = np.load(
    "../data/processed/validation_sequences.npz"
)["X"]

y_val = np.load(
    "../data/processed/validation_sequences.npz"
)["y"]

X_test = np.load(
    "../data/processed/test_sequences.npz"
)["X"]

y_test = np.load(
    "../data/processed/test_sequences.npz"
)["y"]

val_target_mask = np.load(
    "../data/processed/validation_target_mask.npy"
)

test_target_mask = np.load(
    "../data/processed/test_target_mask.npy"
)

print("Validation X:", X_val.shape)
print("Validation y:", y_val.shape)

print("Test X:", X_test.shape)
print("Test y:", y_test.shape)

print("Validation mask:", val_target_mask.shape)
print("Test mask:", test_target_mask.shape)

Validation X: (3410, 12, 207)
Validation y: (3410, 207)
Test X: (6838, 12, 207)
Test y: (6838, 207)
Validation mask: (3410, 207)
Test mask: (6838, 207)


In [10]:
import joblib

scaler = joblib.load(
    "../data/processed/traffic_scaler.pkl"
)

print("Scaler loaded successfully.")

Scaler loaded successfully.


In [11]:
def classify_congestion(speed):
    if speed <= 20:
        return "Severe Congestion"
    elif speed <= 40:
        return "Heavy Congestion"
    elif speed <= 60:
        return "Moderate Traffic"
    else:
        return "Free Flow"

In [12]:
test_speeds = [10, 30, 50, 65]

for speed in test_speeds:
    print(speed, "mph ->", classify_congestion(speed))

10 mph -> Severe Congestion
30 mph -> Heavy Congestion
50 mph -> Moderate Traffic
65 mph -> Free Flow


In [13]:
checkpoint = torch.load(
    "../models/urbanflow_v2_best.pth",
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:")
print(type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")
    print(list(checkpoint.keys()))

Checkpoint type:
<class 'collections.OrderedDict'>

Checkpoint keys:
['spatial_gate', 'temporal_branch.positional_encoding', 'temporal_branch.input_projection.weight', 'temporal_branch.input_projection.bias', 'temporal_branch.transformer.layers.0.self_attn.in_proj_weight', 'temporal_branch.transformer.layers.0.self_attn.in_proj_bias', 'temporal_branch.transformer.layers.0.self_attn.out_proj.weight', 'temporal_branch.transformer.layers.0.self_attn.out_proj.bias', 'temporal_branch.transformer.layers.0.linear1.weight', 'temporal_branch.transformer.layers.0.linear1.bias', 'temporal_branch.transformer.layers.0.linear2.weight', 'temporal_branch.transformer.layers.0.linear2.bias', 'temporal_branch.transformer.layers.0.norm1.weight', 'temporal_branch.transformer.layers.0.norm1.bias', 'temporal_branch.transformer.layers.0.norm2.weight', 'temporal_branch.transformer.layers.0.norm2.bias', 'temporal_branch.transformer.layers.1.self_attn.in_proj_weight', 'temporal_branch.transformer.layers.1.self_a

In [14]:
for name, tensor in checkpoint.items():
    print(f"{name:60s} {tuple(tensor.shape)}")

spatial_gate                                                 ()
temporal_branch.positional_encoding                          (1, 12, 128)
temporal_branch.input_projection.weight                      (128, 1)
temporal_branch.input_projection.bias                        (128,)
temporal_branch.transformer.layers.0.self_attn.in_proj_weight (384, 128)
temporal_branch.transformer.layers.0.self_attn.in_proj_bias  (384,)
temporal_branch.transformer.layers.0.self_attn.out_proj.weight (128, 128)
temporal_branch.transformer.layers.0.self_attn.out_proj.bias (128,)
temporal_branch.transformer.layers.0.linear1.weight          (256, 128)
temporal_branch.transformer.layers.0.linear1.bias            (256,)
temporal_branch.transformer.layers.0.linear2.weight          (128, 256)
temporal_branch.transformer.layers.0.linear2.bias            (128,)
temporal_branch.transformer.layers.0.norm1.weight            (128,)
temporal_branch.transformer.layers.0.norm1.bias              (128,)
temporal_branch.transform

In [15]:
for name, tensor in checkpoint.items():
    if name.startswith("spatial_branch") or name == "spatial_gate":
        print(f"{name:50s} {tuple(tensor.shape)}")

spatial_gate                                       ()
spatial_branch.gcn1.weight                         (128, 1)
spatial_branch.gcn1.bias                           (128,)
spatial_branch.gcn2.weight                         (128, 128)
spatial_branch.gcn2.bias                           (128,)


In [16]:
class TemporalTransformerBranch(nn.Module):

    def __init__(
        self,
        num_sensors,
        hidden_size=128,
        num_heads=4,
        num_layers=2,
        dropout=0.2
    ):
        super().__init__()

        self.num_sensors = num_sensors
        self.hidden_size = hidden_size

        self.input_projection = nn.Linear(
            1,
            hidden_size
        )

        self.positional_encoding = nn.Parameter(
            torch.zeros(
                1,
                12,
                hidden_size
            )
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=num_heads,
            dim_feedforward=hidden_size * 2,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

    def forward(self, x):

        batch_size = x.size(0)


        x = x.permute(0, 2, 1)

        x = x.reshape(
            batch_size * self.num_sensors,
            12,
            1
        )

        x = self.input_projection(x)

        x = x + self.positional_encoding

        x = self.transformer(x)

        x = x[:, -1, :]

        x = x.reshape(
            batch_size,
            self.num_sensors,
            self.hidden_size
        )

        return x


class SpatialGNNBranch(nn.Module):

    def __init__(
        self,
        hidden_size=128
    ):
        super().__init__()

        self.gcn1 = nn.Linear(
            1,
            hidden_size
        )

        self.gcn2 = nn.Linear(
            hidden_size,
            hidden_size
        )

    def graph_convolution(
        self,
        x,
        adjacency,
        layer
    ):

        aggregated = torch.matmul(
            adjacency,
            x
        )

        return layer(aggregated)

    def forward(
        self,
        x,
        adjacency
    ):

        snapshot = x[:, -1, :]

        snapshot = snapshot.unsqueeze(-1)

        h = self.graph_convolution(
            snapshot,
            adjacency,
            self.gcn1
        )

        h = F.relu(h)

        h = self.graph_convolution(
            h,
            adjacency,
            self.gcn2
        )

        h = F.relu(h)

        return h


class UrbanFlowV2(nn.Module):

    def __init__(
        self,
        num_sensors,
        hidden_size=128,
        num_heads=4,
        num_layers=2,
        dropout=0.2
    ):
        super().__init__()

        self.temporal_branch = TemporalTransformerBranch(
            num_sensors=num_sensors,
            hidden_size=hidden_size,
            num_heads=num_heads,
            num_layers=num_layers,
            dropout=dropout
        )

        self.spatial_branch = SpatialGNNBranch(
            hidden_size=hidden_size
        )

        self.spatial_gate = nn.Parameter(
            torch.tensor(0.1)
        )

        self.fusion = nn.Sequential(
            nn.Linear(
                hidden_size * 2,
                hidden_size
            ),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.output_layer = nn.Linear(
            hidden_size,
            1
        )

    def forward(
        self,
        x,
        adjacency
    ):

        temporal_features = self.temporal_branch(x)

        spatial_features = self.spatial_branch(
            x,
            adjacency
        )

        spatial_features = (
            self.spatial_gate *
            spatial_features
        )

        combined = torch.cat(
            [
                temporal_features,
                spatial_features
            ],
            dim=-1
        )

        fused = self.fusion(
            combined
        )

        output = self.output_layer(
            fused
        ).squeeze(-1)

        return output

In [17]:
adjacency_matrix = np.load(
    "../data/processed/adjacency_matrix.npy"
)

A = torch.tensor(
    adjacency_matrix,
    dtype=torch.float32
)

degree = A.sum(dim=1)

degree_inv_sqrt = torch.pow(
    degree.clamp(min=1e-8),
    -0.5
)

D_inv_sqrt = torch.diag(
    degree_inv_sqrt
)

A_normalized = (
    D_inv_sqrt
    @ A
    @ D_inv_sqrt
)

device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

A_normalized = A_normalized.to(device)

print("Adjacency:", A_normalized.shape)
print("Device:", device)

Adjacency: torch.Size([207, 207])
Device: mps


In [18]:
NUM_SENSORS = X_val.shape[2]

urbanflow_v2 = UrbanFlowV2(
    num_sensors=NUM_SENSORS,
    hidden_size=128,
    num_heads=4,
    num_layers=2,
    dropout=0.2
).to(device)

print("Model created.")

Model created.


In [25]:
checkpoint = torch.load(
    "../models/urbanflow_v2_best.pth",
    map_location=device,
    weights_only=True
)

load_result = urbanflow_v2.load_state_dict(
    checkpoint,
    strict=True
)

print("Checkpoint loaded successfully.")
print(load_result)

Checkpoint loaded successfully.
<All keys matched successfully>


In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [26]:
urbanflow_v2.eval()

sample_input = torch.tensor(
    X_val[:8],
    dtype=torch.float32,
    device=device
)

with torch.no_grad():

    sample_prediction = urbanflow_v2(
        sample_input,
        A_normalized
    )

print(
    "Input shape:",
    sample_input.shape
)

print(
    "Prediction shape:",
    sample_prediction.shape
)

print(
    "Prediction range:",
    sample_prediction.min().item(),
    "to",
    sample_prediction.max().item()
)

Input shape: torch.Size([8, 12, 207])
Prediction shape: torch.Size([8, 207])
Prediction range: -8.18211841583252 to 0.8974301218986511


In [27]:

urbanflow_v2.eval()

val_predictions_scaled = []

val_tensor = torch.tensor(
    X_val,
    dtype=torch.float32
)

with torch.no_grad():
    for start in range(0, len(val_tensor), 64):

        batch = val_tensor[start:start + 64].to(device)

        batch_prediction = urbanflow_v2(
            batch,
            A_normalized
        )

        val_predictions_scaled.append(
            batch_prediction.cpu().numpy()
        )

val_predictions_scaled = np.concatenate(
    val_predictions_scaled,
    axis=0
)

print("Validation prediction shape:", val_predictions_scaled.shape)
print(
    "Scaled prediction range:",
    val_predictions_scaled.min(),
    "to",
    val_predictions_scaled.max()
)

Validation prediction shape: (3410, 207)
Scaled prediction range: -8.546597 to 3.2110448


In [28]:

val_predictions_mph = scaler.inverse_transform(
    val_predictions_scaled
)

val_targets_mph = scaler.inverse_transform(
    y_val
)

print("Prediction shape:", val_predictions_mph.shape)
print("Target shape:", val_targets_mph.shape)

print(
    "Prediction range:",
    val_predictions_mph.min(),
    "to",
    val_predictions_mph.max()
)

print(
    "Target range:",
    val_targets_mph.min(),
    "to",
    val_targets_mph.max()
)

Prediction shape: (3410, 207)
Target shape: (3410, 207)
Prediction range: -0.8463669 to 70.85659
Target range: 0.4999962 to 70.0


In [29]:
def classify_congestion(speed):
    if speed <= 20:
        return "Severe Congestion"
    elif speed <= 40:
        return "Heavy Congestion"
    elif speed <= 60:
        return "Moderate Traffic"
    else:
        return "Free Flow"


predicted_categories = np.vectorize(classify_congestion)(
    val_predictions_mph
)

actual_categories = np.vectorize(classify_congestion)(
    val_targets_mph
)

print("Predicted categories shape:", predicted_categories.shape)
print("Actual categories shape:", actual_categories.shape)

Predicted categories shape: (3410, 207)
Actual categories shape: (3410, 207)


In [30]:
validation_target_mask = np.load(
    "../data/processed/validation_target_mask.npy"
)

print("Mask shape:", validation_target_mask.shape)
print("Valid target observations:", validation_target_mask.sum())
print("Total observations:", validation_target_mask.size)

Mask shape: (3410, 207)
Valid target observations: 664844
Total observations: 705870


In [31]:
valid_predicted_categories = predicted_categories[
    validation_target_mask
]

valid_actual_categories = actual_categories[
    validation_target_mask
]

print(
    "Valid predicted categories:",
    len(valid_predicted_categories)
)

print(
    "Valid actual categories:",
    len(valid_actual_categories)
)

Valid predicted categories: 664844
Valid actual categories: 664844


In [32]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

class_labels = [
    "Severe Congestion",
    "Heavy Congestion",
    "Moderate Traffic",
    "Free Flow"
]

accuracy = accuracy_score(
    valid_actual_categories,
    valid_predicted_categories
)

precision = precision_score(
    valid_actual_categories,
    valid_predicted_categories,
    labels=class_labels,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    valid_actual_categories,
    valid_predicted_categories,
    labels=class_labels,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    valid_actual_categories,
    valid_predicted_categories,
    labels=class_labels,
    average="weighted",
    zero_division=0
)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

Accuracy : 0.8189
Precision: 0.8248
Recall   : 0.8189
F1-score : 0.8194


In [33]:
print(
    classification_report(
        valid_actual_categories,
        valid_predicted_categories,
        labels=class_labels,
        zero_division=0
    )
)

                   precision    recall  f1-score   support

Severe Congestion       0.76      0.38      0.51     20467
 Heavy Congestion       0.61      0.63      0.62     48895
 Moderate Traffic       0.62      0.69      0.65    138884
        Free Flow       0.91      0.90      0.91    456598

         accuracy                           0.82    664844
        macro avg       0.73      0.65      0.67    664844
     weighted avg       0.82      0.82      0.82    664844



In [34]:
cm = confusion_matrix(
    valid_actual_categories,
    valid_predicted_categories,
    labels=class_labels
)

cm_df = pd.DataFrame(
    cm,
    index=class_labels,
    columns=class_labels
)

print(cm_df)

                   Severe Congestion  Heavy Congestion  Moderate Traffic  \
Severe Congestion               7787              9525              2400   
Heavy Congestion                1829             30786             13773   
Moderate Traffic                 367              6872             96158   
Free Flow                        211              3038             43624   

                   Free Flow  
Severe Congestion        755  
Heavy Congestion        2507  
Moderate Traffic       35487  
Free Flow             409725  


In [35]:

classification_results = pd.DataFrame({
    "metric": [
        "accuracy",
        "weighted_precision",
        "weighted_recall",
        "weighted_f1",
        "macro_f1"
    ],
    "value": [
        accuracy,
        precision,
        recall,
        f1,
        f1_score(
            valid_actual_categories,
            valid_predicted_categories,
            labels=class_labels,
            average="macro",
            zero_division=0
        )
    ]
})

classification_results.to_csv(
    "../results/congestion_classification_metrics.csv",
    index=False
)

cm_df.to_csv(
    "../results/congestion_confusion_matrix.csv"
)

print("Congestion classification results saved successfully.")

Congestion classification results saved successfully.


In [36]:
report = classification_report(
    valid_actual_categories,
    valid_predicted_categories,
    labels=class_labels,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

report_df.to_csv(
    "../results/congestion_classification_report.csv"
)

print(report_df)

                   precision    recall  f1-score        support
Severe Congestion   0.763881  0.380466  0.507942   20467.000000
Heavy Congestion    0.613010  0.629635  0.621212   48895.000000
Moderate Traffic    0.616575  0.692362  0.652275  138884.000000
Free Flow           0.913598  0.897343  0.905398  456598.000000
accuracy            0.818923  0.818923  0.818923       0.818923
macro avg           0.726766  0.649951  0.671706  664844.000000
weighted avg        0.824836  0.818923  0.819385  664844.000000


In [37]:
actual_category_counts = pd.Series(
    valid_actual_categories
).value_counts()

predicted_category_counts = pd.Series(
    valid_predicted_categories
).value_counts()

category_distribution = pd.DataFrame({
    "actual_count": actual_category_counts,
    "predicted_count": predicted_category_counts
}).fillna(0).astype(int)

category_distribution["actual_percentage"] = (
    category_distribution["actual_count"]
    / len(valid_actual_categories)
    * 100
)

category_distribution["predicted_percentage"] = (
    category_distribution["predicted_count"]
    / len(valid_predicted_categories)
    * 100
)

category_distribution = category_distribution.reindex(
    class_labels
)

print(category_distribution)

                   actual_count  predicted_count  actual_percentage  \
Severe Congestion         20467            10194           3.078467   
Heavy Congestion          48895            50221           7.354357   
Moderate Traffic         138884           155955          20.889712   
Free Flow                456598           448474          68.677464   

                   predicted_percentage  
Severe Congestion              1.533292  
Heavy Congestion               7.553802  
Moderate Traffic              23.457382  
Free Flow                     67.455523  


In [38]:
category_distribution.to_csv(
    "../results/congestion_category_distribution.csv"
)

print("Category distribution saved successfully.")

Category distribution saved successfully.


In [40]:

np.save(
    "../results/urbanflow_v2_validation_predictions_mph.npy",
    val_predictions_mph
)

np.save(
    "../results/validation_targets_mph.npy",
    val_targets_mph
)

np.save(
    "../results/validation_target_mask.npy",
    validation_target_mask
)

print("Validation speed predictions saved successfully.")

Validation speed predictions saved successfully.
